<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/05-Data-Visualization/Untitled28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Step 1: Load and Explore the Spotify 2023 Dataset
# Purpose: Import libraries, load data, understand its shape
# ============================================

# Import our core libraries
import pandas as pd                   # Data manipulation
import matplotlib.pyplot as plt       # Chart building (the foundation)
import seaborn as sns                 # Beautiful charts with less code

# Load the dataset directly from GitHub
# encoding='latin-1' handles special characters in artist names
url = 'https://raw.githubusercontent.com/c-marq/AI-Thinking-CAI1001C/main/05-Data-Visualization/Datasets/spotify-2023.csv'
df = pd.read_csv(url, encoding='latin-1')

# First look: how big is this dataset?
print(f"Dataset shape: {df.shape[0]} songs, {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")

In [ ]:
# ============================================
# Quick data inspection
# ============================================

# Show the first 5 rows
df.head()

In [ ]:
# ============================================
# Check data types - this is where surprises hide
# ============================================

# The 'streams' column is stored as text, not numbers
# We need to fix that before we can do any math with it
print(df['streams'].dtype)

# Convert streams to numeric (some entries may be non-numeric)
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

# Verify the fix worked
print(f"\nStreams column type after fix: {df['streams'].dtype}")
print(f"Average streams per song: {df['streams'].mean():,.0f}")

In [ ]:
# ============================================
# Step 2: Bar Chart: Top 10 Most-Streamed Artists
# Purpose: Compare categories using a horizontal bar chart
# ============================================

# Group by artist, sum their total streams, get top 10
top_artists = (
    df.groupby('artist(s)_name')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Create the figure and set its size
# figsize=(10, 6) means 10 inches wide, 6 inches tall
fig, ax = plt.subplots(figsize=(10, 6))

# Horizontal bar chart - easier to read artist names
# We reverse the order so the highest value is on top
ax.barh(
    top_artists.index[::-1],           # Artist names (reversed for top-down)
    top_artists.values[::-1],          # Stream counts (reversed to match)
    color='#1DB954',                   # Spotify green
    edgecolor='white'
)

# Add labels - a chart without labels is a chart nobody can read
ax.set_xlabel('Total Streams (billions)', fontsize=12)
ax.set_title('Top 10 Most-Streamed Artists on Spotify (2023)', fontsize=14, fontweight='bold')

# Format x-axis to show billions instead of raw numbers
# This makes the chart readable instead of showing 14-digit numbers
ax.set_xticklabels([f'{x/1e9:.1f}B' for x in ax.get_xticks()])

# Remove the top and right borders for a cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Step 3: Histogram: Distribution of BPM (Beats Per Minute)
# Purpose: See the shape of tempo across 953 hit songs
# ============================================

fig, ax = plt.subplots(figsize=(10, 5))

# Create the histogram
# bins=25 divides the BPM range into 25 equal buckets
ax.hist(
    df['bpm'].dropna(),                # Drop any missing BPM values
    bins=25,
    color='#FF6B35',                   # Warm orange
    edgecolor='white',
    alpha=0.85                         # Slight transparency
)

# Add a vertical line at the median BPM
median_bpm = df['bpm'].median()
ax.axvline(x=median_bpm, color='#333333', linestyle='--', linewidth=2, label=f'Median: {median_bpm:.0f} BPM')

# Labels and title
ax.set_xlabel('Beats Per Minute (BPM)', fontsize=12)
ax.set_ylabel('Number of Songs', fontsize=12)
ax.set_title('Tempo Distribution of Top Spotify Songs (2023)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Clean up borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Print a quick summary
print(f"\nBPM Summary:")
print(f"  Slowest song: {df['bpm'].min():.0f} BPM")
print(f"  Fastest song: {df['bpm'].max():.0f} BPM")
print(f"  Median tempo: {median_bpm:.0f} BPM")

In [ ]:
# ============================================
# Step 4: Seaborn Scatter Plot: Danceability vs. Energy
# Purpose: Explore the relationship between two audio features
# ============================================

# Set Seaborn's visual theme - one line upgrades everything
sns.set_theme(style='whitegrid', palette='muted')

fig, ax = plt.subplots(figsize=(10, 6))

# Seaborn scatter plot with color-coded mode (Major vs Minor key)
sns.scatterplot(
    data=df,
    x='danceability_%',
    y='energy_%',
    hue='mode',                        # Color by Major/Minor key
    alpha=0.6,                         # Transparency so overlapping dots are visible
    ax=ax
)

# Labels and title
ax.set_xlabel('Danceability (%)', fontsize=12)
ax.set_ylabel('Energy (%)', fontsize=12)
ax.set_title('Danceability vs. Energy in Top Spotify Songs (2023)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Step 5: The Same Data, Two Different Stories
# Purpose: See how axis manipulation changes perception
# ============================================

# Get the top 5 artists by total streams
top5 = (
    df.groupby('artist(s)_name')['streams']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Chart A: Honest version (Y-axis starts at 0) ---
ax1.bar(range(5), top5.values, color='#1DB954', edgecolor='white')
ax1.set_xticks(range(5))
ax1.set_xticklabels(top5.index, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('Total Streams')
ax1.set_title('Top 5 Artists: Honest Scale', fontweight='bold')
ax1.set_ylim(0, top5.values.max() * 1.15)   # Starts at 0

# --- Chart B: Misleading version (truncated Y-axis) ---
ax2.bar(range(5), top5.values, color='#FF4444', edgecolor='white')
ax2.set_xticks(range(5))
ax2.set_xticklabels(top5.index, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Total Streams')
ax2.set_title('Top 5 Artists: Truncated Scale', fontweight='bold')
ax2.set_ylim(top5.values.min() * 0.92, top5.values.max() * 1.05)  # Starts near minimum

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()